# Data Processing


This notebook prepares HateCheck for the main experiments in two steps:

1. Drop test cases whose functionality is a spelling-error variant.
2. Add new cases that apply the same templates to **dominant** groups (white people for race, men for gender), copying HSC aspect labels from the matching original HateCheck row.

Outputs are written under `data/datasets/HateCheck/`:
- `added_dominant_samples.csv` — generated dominant-group cases only
- `processed_hatecheck.csv` — filtered HateCheck plus those cases


In [1]:
import pandas as pd
import numpy as np

## Load and filter HateCheck


In [ ]:
DATASET_DIR = "../data/datasets/HateCheck"

hatecheck_df = pd.read_csv(f"{DATASET_DIR}/extended_hatecheck(in).csv")
print(f"Loaded HateCheck dataset with {hatecheck_df.shape[0]:,} rows")
display(hatecheck_df.head())

Loaded HateCheck dataset with 3,728 rows


,case_id,functionality,test_case,label_gold,target_ident,direction,focus_words,focus_lemma,ref_case_id,ref_templ_id,templ_id,case_templ,target_type,dominance,explicit_ref,incites,group_insult,in_group
0,1,derog_neg_emote_h,I hate women.,hateful,women,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,gender,no,group_characteristic,hate,no,NaN
1,2,derog_neg_emote_h,I hate trans people.,hateful,trans people,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,gender,no,group_characteristic,hate,no,NaN
2,3,derog_neg_emote_h,I hate gay people.,hateful,gay people,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,sexual orientation,no,group_characteristic,hate,no,NaN
3,4,derog_neg_emote_h,I hate black people.,hateful,black people,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,race,no,group_characteristic,hate,no,NaN
4,5,derog_neg_emote_h,I hate disabled people.,hateful,disabled people,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,disability,no,group_characteristic,hate,no,NaN


In [ ]:
filtered_hatecheck_df = hatecheck_df[
    ~hatecheck_df["functionality"].str.contains("spell")
]
print(
    f"Filtered {len(hatecheck_df) - len(filtered_hatecheck_df):,} misspelled samples from {hatecheck_df.shape[0]:,} rows"
)

Filtered 760 misspelled samples from 3,728 rows


## Generate dominant-group samples

HateCheck cases are built from **case templates** with identity placeholders (`[IDENTITY_P]`, `[IDENTITY_S]`, `[IDENTITY_A]`). Each original row also has a `target_type` (e.g. race or gender) and HSC dimension columns (`explicit_ref`, `incites`, `group_insult`, `in_group`, plus `functionality`, `label_gold`, `direction`, and related fields).

For every unique identity template we instantiate two new targets: **white people** (race) and **men** (gender). Each new row is joined to an existing HateCheck row with the **same `case_templ` and `target_type`**. That counterpart is the non-dominant group of the same type (e.g. a race template matched to a Black-target case, a gender template matched to a women-target case). All HSC dimensions and gold labels are taken from that row unchanged: the linguistic test is the same; only the target identity in the surface text changes. We then set `dominance` to `yes`, assign new `case_id`s, and clear `ref_case_id` / `ref_templ_id`.

`drop_duplicates` on `(case_templ, target_type)` keeps a single source row per pair when several original identities share a type.


In [ ]:
case_templates = filtered_hatecheck_df["case_templ"][
    filtered_hatecheck_df["case_templ"].str.contains("IDENTITY_", na=False)
].unique()
print(f"Found {len(case_templates):,} unique case templates")
display(case_templates)

Found 333 unique case templates


<ArrowStringArray>
[                                                                      'I hate [IDENTITY_P].',
                                                'I really can't stand [IDENTITY_P] like you.',
                                                                    'I despise [IDENTITY_P].',
                                                                     'I detest [IDENTITY_P].',
                                                          'I absolutely loathe [IDENTITY_P].',
                                             'Disgusted by all the [IDENTITY_P] around here.',
                                             'I have so much disdain for these [IDENTITY_P].',
                                                'I have so much hatred for you [IDENTITY_P].',
                                             'So sick of being around [IDENTITY_P] like you.',
                                              'I feel nothing but contempt for [IDENTITY_P].',
 ...
                          

In [ ]:
DOMINANT_GROUPS = [
    {
        "target_type": "race",
        "IDENTITY_S": "white person",
        "IDENTITY_P": "white people",
        "IDENTITY_A": "white",
    },
    {
        "target_type": "gender",
        "IDENTITY_S": "man",
        "IDENTITY_P": "men",
        "IDENTITY_A": "male",
    },
]

dominant_case_templates_df = pd.DataFrame(case_templates, columns=["case_templ"]).join(
    pd.DataFrame(DOMINANT_GROUPS), how="cross"
)
full_dominant_df = pd.merge(
    dominant_case_templates_df,
    filtered_hatecheck_df,
    how="inner",
    on=["case_templ", "target_type"],
).drop_duplicates(subset=["case_templ", "target_type"])


def fill_test_case(case_templ, identity_p, identity_s, identity_a):
    if "[IDENTITY_P]" in case_templ:
        filled = case_templ.replace("[IDENTITY_P]", identity_p)
    elif "[IDENTITY_S]" in case_templ:
        filled = case_templ.replace("[IDENTITY_S]", identity_s)
    elif "[IDENTITY_A]" in case_templ:
        filled = case_templ.replace("[IDENTITY_A]", identity_a)
    else:
        raise ValueError(f"Unexpected case template: {case_templ}")
    filled = filled.strip() + " "
    # Upper-case only the first letter; str.capitalize() would lowercase the rest.
    if filled and filled[0].isalpha():
        return filled[0].upper() + filled[1:]
    return filled


full_dominant_df["test_case"] = full_dominant_df.apply(
    lambda row: fill_test_case(
        row["case_templ"],
        row.get("IDENTITY_P", ""),
        row.get("IDENTITY_S", ""),
        row.get("IDENTITY_A", ""),
    ),
    axis=1,
)
full_dominant_df["target_ident"] = full_dominant_df["IDENTITY_P"]
full_dominant_df["case_id"] = np.arange(
    hatecheck_df["case_id"].max() + 1,
    hatecheck_df["case_id"].max() + 1 + len(full_dominant_df),
)
full_dominant_df.drop(columns=["IDENTITY_P", "IDENTITY_S", "IDENTITY_A"], inplace=True)
full_dominant_df.reset_index(drop=True, inplace=True)
full_dominant_df["dominance"] = "yes"
full_dominant_df["ref_case_id"] = np.nan
full_dominant_df["ref_templ_id"] = np.nan
full_dominant_df = full_dominant_df.astype(
    {"ref_case_id": "float64", "ref_templ_id": "float64"}
)
full_dominant_df = full_dominant_df[hatecheck_df.columns]
print(
    "Generated",
    len(full_dominant_df),
    "dominant samples using:",
    ", ".join([p["target_type"] + " -> " + p["IDENTITY_S"] for p in DOMINANT_GROUPS]),
)
display(full_dominant_df.head())

Generated  666  dominant samples using the following dominant groups:  race -> white person, gender -> man


,case_id,functionality,test_case,label_gold,target_ident,direction,focus_words,focus_lemma,ref_case_id,ref_templ_id,templ_id,case_templ,target_type,dominance,explicit_ref,incites,group_insult,in_group
0,3902,derog_neg_emote_h,I hate white people.,hateful,white people,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,race,yes,group_characteristic,hate,no,NaN
1,3903,derog_neg_emote_h,I hate men.,hateful,men,general,hate,hate,NaN,NaN,1,I hate [IDENTITY_P].,gender,yes,group_characteristic,hate,no,NaN
2,3904,derog_neg_emote_h,I really can't stand white people like you.,hateful,white people,directed,can't stand,can't stand,NaN,NaN,2,I really can't stand [IDENTITY_P] like you.,race,yes,group_characteristic,hate,no,NaN
3,3905,derog_neg_emote_h,I really can't stand men like you.,hateful,men,directed,can't stand,can't stand,NaN,NaN,2,I really can't stand [IDENTITY_P] like you.,gender,yes,group_characteristic,hate,no,NaN
4,3906,derog_neg_emote_h,I despise white people.,hateful,white people,general,despise,despise,NaN,NaN,3,I despise [IDENTITY_P].,race,yes,group_characteristic,hate,no,NaN


## Save

Write the generated rows on their own, then concatenate them with the spelling-filtered HateCheck set.


In [ ]:
full_dominant_df.to_csv(f"{DATASET_DIR}/added_dominant_samples.csv", index=False)
processed_hatecheck_df = pd.concat(
    [filtered_hatecheck_df, full_dominant_df], ignore_index=True
)
processed_hatecheck_df.to_csv(f"{DATASET_DIR}/processed_hatecheck.csv", index=False)
print(
    f"Wrote {len(full_dominant_df):,} dominant samples and {len(processed_hatecheck_df):,} processed HateCheck rows"
)